# Wrapper Functions & Decorators — Practice Notebook

This notebook walks through wrapper functions step‑by‑step, then finishes with the `@decorator` syntax.

## 1) Understand a wrapper function (basic)
A *wrapper function* is a function that defines and returns another function (often called `inner`). The inner function usually does **extra work** before/after calling the original function.

Goal: print `this is wrapper` every time our wrapped function runs.

In [ ]:
def wrapper(func):
    def inner():
        print("this is wrapper")
        func()
    return inner

def greet():
    print("hello!")

wrapped_greet = wrapper(greet)
wrapped_greet()  # Expect: prints 'this is wrapper' then 'hello!'


## 2) Wrapper to measure execution time
We can add timing logic around `func()` using `time.perf_counter()`.

In [ ]:
import time

def time_wrapper(func):
    def inner():
        start = time.perf_counter()
        try:
            retval = func()
            return retval
        finally:
            end = time.perf_counter()
            print(f"time: {end - start:.6f} seconds")
    return inner

def slow_work():
    total = 0
    for i in range(10_000_00):  # ~1e6 loop
        total += i
    return total

timed_slow_work = time_wrapper(slow_work)
result = timed_slow_work()
print("result:", result)


## 3) Pass parameters through the wrapper
Use `*args` and `**kwargs` to forward any positional/keyword arguments to the wrapped function.

In [ ]:
def wrapper_with_args(func):
    def inner(*args, **kwargs):
        print("this is wrapper (args forwarded)")
        retval = func(*args, **kwargs)
        return retval
    return inner

def add(a, b, scale=1):
    return scale * (a + b)

wrapped_add = wrapper_with_args(add)
print(wrapped_add(3, 4))               # 7
print(wrapped_add(3, 4, scale=10))     # 70


## 4) Return a value from the inner function
`inner` should return the result of `func(*args, **kwargs)` so callers receive the original function's value.

In [ ]:
def wrapper_returns_value(func):
    def inner(*args, **kwargs):
        print("this is wrapper (will return value)")
        value = func(*args, **kwargs)
        print("wrapped function returned:", value)
        return value  # <-- return to caller
    return inner

def multiply(a, b):
    return a * b

wrapped_multiply = wrapper_returns_value(multiply)
out = wrapped_multiply(6, 7)
print("final:", out)


## 5) Decorator (`@`) syntax
Decorators are just a nicer way to apply a wrapper. `@decorator_name` above a function is equivalent to `func = decorator_name(func)`.

### The first decorator example

In [ ]:
def wrapper(func):
    def inner():
        print("this is wrapper")
        func()
    return inner

def hello():
    print("hello!")

wrapped_hello = wrapper(hello)
wrapped_hello()

In [ ]:
def simple_decorator(func):
    def inner():
        print("this is wrapper")
        return func()
    return inner

@simple_decorator
def hello():
    print("hello!")

hello()

### 2) Decorator Example 2


In [ ]:
import time
from functools import wraps

def decorator(func):
    @wraps(func)  # keeps original __name__, __doc__, etc.
    def inner(*args, **kwargs):
        print("this is wrapper (decorator)")
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            end = time.perf_counter()
            print(f'Elapsed: {end - start:.6f} seconds')
    return inner

@decorator
def fib(n):
    """Return nth Fibonacci number (naive)."""
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print("fib(10) ->", fib(10))

### (Optional) Combine multiple decorators
Order matters: the top decorator runs last around the function call.

In [ ]:
from functools import wraps

def logger(func):
    @wraps(func)
    def inner(*args, **kwargs):
        print(f"[logger] calling {func.__name__}{args, kwargs}")
        value = func(*args, **kwargs)
        print(f"[logger] returned {value}")
        return value
    return inner

def ensure_int(func):
    @wraps(func)
    def inner(*args, **kwargs):
        value = func(*args, **kwargs)
        if not isinstance(value, int):
            raise TypeError("return value must be int")
        return value
    return inner

@ensure_int
@logger
def sum_to(n):
    return sum(range(n+1))

print("sum_to(5) ->", sum_to(5))


## Exercises
Complete each exercise **in its own code cell** below. After you finish an exercise, **Save and pin a revision** and name it `Exercise N` (Colab: *File → Save and pin revision*). Each exercise is graded as a separate item, so do them one at a time.


### Exercise 1 — A basic wrapper (warm-up)
Write `shout_wrapper(func)` that prints `=== START ===` **before** and `=== END ===` **after** calling `func()`. Use it to wrap a `greet()` function that prints `hello!`.


In [ ]:
def shout_wrapper(func):
    # TODO: define inner() that prints START, calls func(), prints END
    pass

def greet():
    print("hello!")

# TODO: wrap greet with shout_wrapper and call it


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 1"**


### Exercise 2 — Forward and print keyword arguments
Modify `wrapper_with_args` so that `inner` **also prints the keyword arguments** it received before forwarding them to `func`.


In [ ]:
def wrapper_with_args(func):
    def inner(*args, **kwargs):
        # TODO: print kwargs, then forward args+kwargs to func and return its value
        pass
    return inner

@wrapper_with_args
def add(a, b, scale=1):
    return scale * (a + b)

# TODO: call add(3, 4, scale=10)


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 2"**


### Exercise 3 — @count_calls
Write a decorator `@count_calls` that counts how many times the decorated function has been called and prints the running count on each call.


In [ ]:
def count_calls(func):
    # TODO: keep a counter; print it each call; return func's result
    pass

@count_calls
def say_hi():
    print("hi")

# TODO: call say_hi() three times


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 3"**


### Exercise 4 — @retry(k)
Write a **parameterized** decorator `@retry(k)` that retries the function up to `k` times if it raises an exception. If all attempts fail, re-raise the last exception.


In [ ]:
def retry(k):
    def decorator(func):
        # TODO: try func up to k times; return on success; re-raise if all fail
        pass
    return decorator

@retry(3)
def flaky():
    raise ValueError("boom")  # change later to test the success path

# TODO: call flaky() inside a try/except and print what happens


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 4"**


### Exercise 5 — @timer
Write a decorator `@timer` that prints how long the function took, using `time.perf_counter()`. It must return the function's value unchanged.


In [ ]:
import time

def timer(func):
    # TODO: time the call, print elapsed seconds, return the value
    pass

@timer
def slow_sum(n):
    return sum(range(n))

# TODO: call slow_sum(1_000_000)


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 5"**


### Exercise 6 — @uppercase
Write a decorator `@uppercase` that converts a **string return value** to uppercase. Assume the wrapped function returns a string.


In [ ]:
def uppercase(func):
    # TODO: call func, uppercase its string result, return it
    pass

@uppercase
def name():
    return "ada lovelace"

# TODO: print name()


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 6"**


### Exercise 7 — @require_positive
Write a decorator `@require_positive` that raises `ValueError` if **any positional argument** is negative; otherwise it calls the function normally.


In [ ]:
def require_positive(func):
    # TODO: check args; raise ValueError on a negative; else call func
    pass

@require_positive
def area(w, h):
    return w * h

# TODO: try area(3, 4) and area(-1, 4)


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 7"**


### Exercise 8 — @cache (and compare with lru_cache)
Write a caching decorator `@cache` for a pure function like `fib`. Store computed results in a dict keyed by the arguments. Then compare your version with `functools.lru_cache`.


In [ ]:
def cache(func):
    # TODO: keep a dict; return stored result if present, else compute+store
    pass

@cache
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

# TODO: print fib(30); then try functools.lru_cache on a second copy and compare


---

👉 Please **(ctrl-s or cmd-s)** save and **"pin revision"**

👉 Rename the current pinned revision to **"Exercise 8"**
